# GA1 (gibberellin) pocket exploration -- per protein, per pose cluster

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`, plus `pliparser`
installed editable from `/Users/ereboul/projects/python-pliparser` for the
atom-level section below)

Explores, for each GA1-docked protein and each ligand-pose cluster
(`ca_cluster`/`ligand_pose_cluster` -- see `scripts/cluster_conformations.py`
and `rescoring/README.md`'s pose-cluster breakdown), three complementary
views of the binding pocket:

- **Residue (AA) contribution** -- mean Rosetta REF2015 ligand<->residue
  two-body energy per **raw protein residue number** (not the CDD/InterPro
  35-position subset -- see note below).
- **Interaction type** -- which kinds of physical interaction PLIP actually
  detects at each residue.
- **Atom-level interaction matrices** -- one matrix per PLIP interaction
  type, protein residue number x ligand atom name, so you can see e.g.
  *which* ligand atom each residue's hydrogen bond is actually to.

**Why raw residue numbers, not CDD "position" 1-35:** the CDD-defined
pocket doesn't fully agree with what PLIP actually finds in contact (see
`rescoring/results/plip_cdd_agreement.csv` -- pooled precision/recall well
under 1.0), so restricting this notebook to the 35 CDD positions would
silently hide any real cluster-to-cluster difference happening outside
that definition. Every plot here uses the complex's own original PDB
residue numbering directly, with **no CDD filtering** -- Rosetta's
`prot_resi` and PLIP's `resnr` are already the same numbering space (see
`plip_analysis.py`'s module docstring: neither `pose_prep.py` nor
ChimeraX's minimization renumbers the protein chain).

Standalone exploratory notebook, not part of the automated Snakemake
pipeline -- run after the rescoring + ChimeraX/PLIP stages (9-16) have
produced `results/all_contacts.csv`, `results/plip_contacts.csv`, and
`results/plip/*_report.txt` + `results/chimerax_minimized/*.pdb` (needed
for the atom-level section only).


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path("..")
RESCORING_ROOT = ROOT / "rescoring"
RESULTS = RESCORING_ROOT / "results"
DATA = RESCORING_ROOT / "data"
PLIP_REPORT_DIR = RESULTS / "plip"
CHIMERAX_MIN_DIR = RESULTS / "chimerax_minimized"

LIGAND = "GA1"

# Shared PLIP interaction-type palette -- used by the interaction-type
# frequency plot below, and to border-color "stabilizing" residues in the
# Rosetta bar plots with the PLIP contact type detected there (see
# plot_residue_contribution / BOND_BORDER_COLORS). Defined here (rather than
# alongside plot_interaction_types) so it's available before that cell runs.
INTERACTION_COLORS = {
    "hydrogen_bonds": "#1f77b4",
    "salt_bridges": "#d62728",
    "hydrophobic_interactions": "#2ca02c",
    "pi_stacks": "#9467bd",
    "pi_cation_interactions": "#8c564b",
    "halogen_bonds": "#e377c2",
    "water_bridges": "#17becf",
    "metal_complexes": "#bcbd22",
}


## Load data

Both tables get `ca_cluster`/`ligand_pose_cluster` freshly joined in from
`manifest.csv` (same as `relax_effect_report.py`'s `add_pose_cluster_labels()`)
rather than trusting `all_contacts.csv`'s own embedded copy of those two
columns -- **that copy is stale for most rows** (per-complex CSVs are
written once by `run_complex.py` and never touched again by later,
resumable reruns, so any complex first scored before
`make_manifest.py --all` was fixed to populate `ligand_pose_cluster`
still carries the old blank value baked in, even though the current
`manifest.csv` has the right one). Re-joining fresh sidesteps that
entirely; dropping the embedded columns first makes sure the merge below
can't silently produce `_x`/`_y` duplicates.

No `position`/CDD filtering here -- every residue Rosetta or PLIP actually
reported is kept, indexed by its own `resnr`.


In [ ]:
manifest = pd.read_csv(DATA / "manifest.csv", usecols=["complex_id", "ca_cluster", "ligand_pose_cluster"])


def _load_ligand_table(csv_name: str, resnr_col: str) -> pd.DataFrame:
    df = pd.read_csv(RESULTS / csv_name)
    df = df[df["ligand"] == LIGAND].drop(columns=["ca_cluster", "ligand_pose_cluster"], errors="ignore")
    df = df.merge(manifest, on="complex_id", how="inner")
    df = df.dropna(subset=["ca_cluster", "ligand_pose_cluster"]).copy()
    df["resnr"] = df[resnr_col].astype(int)
    df["pose_label"] = "ca" + df["ca_cluster"].astype(int).astype(str) + "_pose" + df["ligand_pose_cluster"].astype(int).astype(str)
    return df


rosetta = _load_ligand_table("all_contacts.csv", resnr_col="prot_resi")
plip = _load_ligand_table("plip_contacts.csv", resnr_col="resnr")

GA1_PROTEINS = sorted(rosetta["protein"].unique())

print(f"Rosetta: {len(rosetta)} rows, {rosetta['complex_id'].nunique()} complexes, {len(GA1_PROTEINS)} proteins")
print(f"PLIP:    {len(plip)} rows, {plip['complex_id'].nunique()} complexes, {plip['protein'].nunique()} proteins")
print(GA1_PROTEINS)


## Residue (AA) contribution

One **bar plot per pose cluster** (not a single per-protein heatmap): x =
raw protein residue number, y = mean Rosetta two-body total (REU) across
that pose cluster's complexes, bar color also encodes that value (RdBu_r,
centered at 0). One `twobody_total` per (complex_id, replica, resnr) is
used (it's repeated across each `scoretype` row in `all_contacts.csv` --
see `aggregate.py`'s `residue_rank()`), so it's deduped before averaging.
Residues Rosetta never puts in contact for a given protein simply don't
appear as a bar -- nothing is padded out to every residue in the sequence.

**Why per-cluster bar plots, not one shared heatmap:** a genuinely bad pose
can produce a residue contact with an arbitrarily unfavorable (very
positive) `twobody_total` -- a steric clash has no floor -- while a *good*
pose's most favorable contact is bounded (real binding energies don't go
arbitrarily negative). Putting every pose cluster on one shared heatmap
color scale lets a single clashing pose in one cluster stretch that scale
so far that every other, reasonable cluster reads as a flat wash of
near-zero color, burying the real signal. Each cluster below instead gets
its own bar plot with its own independent y-axis/color range, and any
cluster containing a residue at or above `ANOMALY_TWOBODY_REU` REU is
flagged directly in its title (rather than being silently averaged into a
shared scale) so it reads as "this pose cluster is a clash" rather than as
a fair energetic comparison to the reasonable clusters.

**Bar borders -- PLIP bond type:** every bar for a *stabilizing* residue
(mean `twobody_total` < 0) whose residue PLIP also flags as a contact gets
a colored border in the same palette as the interaction-type plot below --
green for `hydrophobic_interactions`, red for `salt_bridges`, and orange
for a residue where PLIP detects both at once. Bars with no border are
either unfavorable (not stabilizing) or have no PLIP hydrophobic/salt-bridge
contact recorded there; other PLIP contact types (pi-stacking, pi-cation,
...) aren't bordered here but remain visible in the interaction-type plot.


In [ ]:
def residue_contribution_table(protein: str) -> pd.DataFrame:
    """Mean Rosetta two-body total per (pose_label, resnr) for one protein."""
    sub = rosetta[rosetta["protein"] == protein]
    per_edge = sub.drop_duplicates(["complex_id", "replica", "resnr"])
    return (
        per_edge.groupby(["pose_label", "resnr"])
        .agg(mean_twobody=("twobody_total", "mean"), n=("twobody_total", "count"), restype=("prot_resn", "first"))
        .reset_index()
    )


# Mean two-body REU above which a residue contact is treated as a steric
# clash rather than a real interaction, for flagging only (see markdown
# above). Real contacts -- favorable or mildly unfavorable -- sit within
# roughly +/-5 REU under ref2015; a mean reaching double digits reflects
# clash-level atomic overlap, not a comparable binding contribution.
ANOMALY_TWOBODY_REU = 10.0

# Bar-border palette for annotating *stabilizing* residues (mean_twobody < 0)
# with the PLIP contact type detected there -- reuses INTERACTION_COLORS'
# own hydrophobic/salt-bridge colors so this reads consistently with the
# interaction-type plot below, plus one extra color for a residue where PLIP
# detects both at once.
BOND_BORDER_COLORS = {
    "hydrophobic_interactions": INTERACTION_COLORS["hydrophobic_interactions"],
    "salt_bridges": INTERACTION_COLORS["salt_bridges"],
    "mixed": "#ff7f0e",
}
BOND_BORDER_LEGEND_LABELS = {
    "hydrophobic_interactions": "hydrophobic (PLIP)",
    "salt_bridges": "salt bridge (PLIP)",
    "mixed": "hydrophobic + salt bridge (PLIP)",
}


def _stabilizing_bond_types(protein: str, pose_label: str) -> dict[int, str]:
    """resnr -> 'hydrophobic_interactions' | 'salt_bridges' | 'mixed', for
    every residue where PLIP detected that contact type in at least one of
    this pose cluster's complexes. Only hydrophobic/salt-bridge presence is
    tracked -- the two contact types most directly responsible for a
    genuinely stabilizing Rosetta two-body score -- so pi-stacking/cation
    and other PLIP contact types are left unbordered here (still visible in
    the interaction-type plot below). Reads `plip` directly rather than
    reusing interaction_type_table's frequency aggregation, since only
    presence/absence per residue is needed here, not a frequency."""
    sub = plip[(plip["protein"] == protein) & (plip["pose_label"] == pose_label)]
    if sub.empty:
        return {}
    exploded = sub.assign(interaction_type=sub["interaction_types"].str.split(";")).explode("interaction_type")
    exploded = exploded[exploded["interaction_type"].isin(["hydrophobic_interactions", "salt_bridges"])]
    bond_types: dict[int, str] = {}
    for resnr, types in exploded.groupby("resnr")["interaction_type"].unique().items():
        types = set(types)
        bond_types[resnr] = "mixed" if len(types) > 1 else next(iter(types))
    return bond_types


def plot_residue_contribution(protein: str):
    table = residue_contribution_table(protein)
    if table.empty:
        print(f"{protein}: no mapped Rosetta contacts")
        return
    for pose_label in sorted(table["pose_label"].unique()):
        sub = table[table["pose_label"] == pose_label].sort_values("resnr")
        worst = sub["mean_twobody"].max()
        flag = (
            f"  ⚠ ANOMALOUS -- max {worst:.1f} REU >= {ANOMALY_TWOBODY_REU:.0f} REU (likely clash)"
            if worst >= ANOMALY_TWOBODY_REU else ""
        )

        bond_types = _stabilizing_bond_types(protein, pose_label)
        bond_by_resnr = {
            resnr: bond_types.get(resnr)
            for resnr, mean_tb in zip(sub["resnr"], sub["mean_twobody"])
            if mean_tb < 0 and resnr in bond_types
        }
        border_colors = [BOND_BORDER_COLORS.get(bond_by_resnr.get(resnr), "rgba(0,0,0,0)") for resnr in sub["resnr"]]
        border_widths = [0 if c == "rgba(0,0,0,0)" else 3 for c in border_colors]

        fig = px.bar(
            sub, x="resnr", y="mean_twobody", color="mean_twobody",
            color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
            labels=dict(mean_twobody="mean two-body (REU)", resnr="protein residue number"),
            title=f"{protein} -- {pose_label}{flag}",
        )
        fig.update_traces(marker_line_color=border_colors, marker_line_width=border_widths)

        # Border color isn't part of px's own legend (only the continuous
        # REU colorbar, hidden below) -- add one dummy legend marker per
        # bond category actually bordered in this cluster's plot.
        for category in sorted(set(bond_by_resnr.values()) - {None}):
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(size=10, color="white", line=dict(color=BOND_BORDER_COLORS[category], width=3)),
                name=BOND_BORDER_LEGEND_LABELS[category], showlegend=True,
            ))

        fig.update_xaxes(title="protein residue number", type="category")
        fig.update_yaxes(title="mean two-body (REU)")
        fig.update_layout(coloraxis_showscale=False)
        fig.show()


## Unfavorable residue decomposition

For every residue flagged as net-*unfavorable* above (mean `twobody_total`
> 0 REU -- a bad, not stabilizing, contact), one stacked bar plot per pose
cluster showing what that positive total is actually made of: Rosetta's
own per-scoretype terms (`fa_rep`, `fa_atr`, `fa_sol`, `fa_elec`,
`lk_ball_wtd`, `hbond_sc`, `hbond_bb_sc` -- `all_contacts.csv`'s
`weighted_energy` column, still per (complex_id, replica, resnr,
scoretype), averaged the same way `residue_contribution_table` averages
`twobody_total`). A residue dominated by `fa_rep` (repulsive van der Waals)
is a steric clash; one dominated by `fa_elec` or `fa_sol` reflects a bad
electrostatic/desolvation mismatch instead -- two very different problems
that a single `twobody_total` number can't distinguish. Clusters with no
unfavorable residues are skipped entirely.


In [ ]:
SCORETYPE_COLORS = {
    "fa_atr": "#2ca02c",
    "fa_rep": "#d62728",
    "fa_sol": "#ff7f0e",
    "fa_elec": "#1f77b4",
    "lk_ball_wtd": "#9467bd",
    "hbond_sc": "#17becf",
    "hbond_bb_sc": "#8c564b",
}
SCORETYPE_ORDER = list(SCORETYPE_COLORS)


def scoretype_decomposition_table(protein: str) -> pd.DataFrame:
    """Mean weighted_energy per (pose_label, resnr, scoretype) for one
    protein -- the same all_contacts.csv rows residue_contribution_table
    sums into twobody_total, kept split by scoretype here instead."""
    sub = rosetta[rosetta["protein"] == protein]
    per_edge = sub.drop_duplicates(["complex_id", "replica", "resnr", "scoretype"])
    return (
        per_edge.groupby(["pose_label", "resnr", "scoretype"])["weighted_energy"]
        .mean()
        .reset_index()
    )


def plot_positive_residue_decomposition(protein: str):
    totals = residue_contribution_table(protein)
    if totals.empty:
        print(f"{protein}: no mapped Rosetta contacts")
        return
    decomposed = scoretype_decomposition_table(protein)
    for pose_label in sorted(totals["pose_label"].unique()):
        bad_resnr = totals[(totals["pose_label"] == pose_label) & (totals["mean_twobody"] > 0)]["resnr"]
        if bad_resnr.empty:
            continue
        sub = decomposed[(decomposed["pose_label"] == pose_label) & (decomposed["resnr"].isin(bad_resnr))]
        fig = px.bar(
            sub.sort_values("resnr"), x="resnr", y="weighted_energy", color="scoretype",
            color_discrete_map=SCORETYPE_COLORS, category_orders=dict(scoretype=SCORETYPE_ORDER),
            labels=dict(weighted_energy="mean weighted energy (REU)", resnr="protein residue number"),
            title=f"{protein} -- {pose_label} -- unfavorable residue decomposition ({len(bad_resnr)} residue(s))",
        )
        fig.add_hline(y=0, line_color="black", line_width=1)
        fig.update_xaxes(title="protein residue number", type="category")
        fig.update_yaxes(title="mean weighted energy (REU)")
        fig.show()


## Interaction type

For each (pose cluster, residue), what fraction of that pose cluster's
complexes show each PLIP interaction type there. `interaction_types` is a
`;`-joined string per (complex, residue) row (a residue can make more than
one kind of contact at once) -- exploded before counting.


In [ ]:
def interaction_type_table(protein: str) -> pd.DataFrame:
    """One row per (pose_label, resnr, interaction_type): fraction of that
    pose cluster's complexes showing that interaction type at that residue."""
    sub = plip[plip["protein"] == protein]
    if sub.empty:
        return sub
    exploded = sub.assign(interaction_type=sub["interaction_types"].str.split(";")).explode("interaction_type")
    per_complex = exploded.drop_duplicates(["complex_id", "pose_label", "resnr", "interaction_type"])
    totals = sub.drop_duplicates(["complex_id", "pose_label"]).groupby("pose_label")["complex_id"].nunique()
    counts = per_complex.groupby(["pose_label", "resnr", "interaction_type"]).size().reset_index(name="n")
    counts["n_complexes_total"] = counts["pose_label"].map(totals)
    counts["frequency"] = counts["n"] / counts["n_complexes_total"]
    return counts


def plot_interaction_types(protein: str):
    counts = interaction_type_table(protein)
    if counts.empty:
        print(f"{protein}: no PLIP contacts")
        return
    fig = px.bar(
        counts, x="resnr", y="frequency", color="interaction_type",
        facet_row="pose_label", color_discrete_map=INTERACTION_COLORS,
        title=f"{protein} -- PLIP interaction type frequency by pose cluster "
              "(y = fraction of that pose cluster's complexes)",
        labels=dict(frequency=""),
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # "pose_label=ca0_pose0" -> "ca0_pose0"
    fig.update_layout(height=180 * counts["pose_label"].nunique() + 120, margin=dict(l=60))
    fig.update_xaxes(type="category")
    fig.show()


## Combined per-residue summary

Same (pose cluster, residue) rows as the bar plots above, with each
residue's top PLIP interaction type(s) attached -- a quick way to read off
"residue 83 is stabilizing (-3.1 REU) via hydrophobic contacts" in one table.


In [ ]:
def combined_summary(protein: str) -> pd.DataFrame:
    rose = residue_contribution_table(protein)
    inter = interaction_type_table(protein)
    if inter.empty:
        rose = rose.copy()
        rose["interaction_type"] = "(none detected)"
        return rose.sort_values(["pose_label", "mean_twobody"])
    top_types = (
        inter.sort_values("frequency", ascending=False)
        .groupby(["pose_label", "resnr"])["interaction_type"]
        .apply(lambda s: ", ".join(s.head(3)))
        .reset_index()
    )
    merged = rose.merge(top_types, on=["pose_label", "resnr"], how="left")
    merged["interaction_type"] = merged["interaction_type"].fillna("(none detected)")
    return merged.sort_values(["pose_label", "mean_twobody"])


## Atom-level interaction matrices (protein residue x ligand atom)

One matrix per PLIP interaction type, per protein: rows = protein residue
number, columns = the ligand's own canonical atom name (`C1`..`C19`,
`O1`..`O6` for GA1 -- assigned once from the SMILES template, in
`ligand_fix.build_template`, and reused unchanged across every pose/backend
-- see that module's docstring), color = fraction of that protein's
complexes showing that specific (residue, atom, interaction type) triple.

Needs atom-level detail `plip_contacts.csv` doesn't carry (it's already
pooled to residue level), so this re-parses the raw
`results/plip/<complex_id>_report.txt` files directly via `pliparser`.
Every PLIP interaction-type table reports the exact 3-D coordinate of the
ligand atom/group involved (`LIGCOO`) -- its own atom-index columns
(`LIGCARBONIDX`, `LIG_IDX_LIST`, `DONORIDX`, ...) turned out to be
PLIP-internal indices, **not** the minimized PDB's own atom serial numbers
(checked by hand: `LIGCARBONIDX=5012` does not correspond to serial 5012
in the PDB), so atom identity is resolved by matching `LIGCOO` back to the
minimized PDB's own chain-`L` HETATM coordinates instead (confirmed exact
to 3 decimal places by hand on a real interaction).

That only works for genuinely atom-to-atom interaction types
(`hydrophobic_interactions` -- confirmed clean, no unmatched coordinates
across the whole corpus). The others PLIP actually reports here --
`salt_bridges`, `pi_cation_interactions`, `pi_stacks` -- are between a
protein atom and a whole ligand **group** (a carboxylate, an aromatic
ring, ...), so `LIGCOO` is that group's centroid, not any single atom's
position, and coordinate-matching it naturally finds nothing. For those,
the ligand-side label falls back to PLIP's own `LIG_GROUP` field
(`"Carboxylate"`, `"Aromatic"`, ...) where present, or a fixed
`"aromatic_ring"` label for `pi_stacks` (which never carries `LIG_GROUP`
but is always a ring by definition) -- both more informative than the
individual-atom axis for a group interaction anyway.

**Note on interaction types actually present**: across every ligand in
this corpus (not just GA1), PLIP only ever reports
`hydrophobic_interactions`, `salt_bridges`, `pi_cation_interactions`, and
`pi_stacks` -- **never** `hydrogen_bonds`, despite these ligands having
plenty of donor/acceptor groups. Worth a closer look separately (possibly
an interaction between the `--nohydro` flag in `plip_run_batch.py` and
exactly how ChimeraX places polar hydrogens) -- not investigated further
here.

This re-reads one report + one PDB per complex, so it's noticeably slower
than the cells above (a few tens of seconds per protein) -- scoped to one
protein at a time rather than pre-computed for all 8.


In [ ]:
from pliparser.plip2csv import plip2dictlist


def _ligand_atom_names(complex_id: str) -> dict[tuple[float, float, float], str]:
    """(x, y, z) rounded to 2dp -> canonical ligand atom name, read straight
    off the minimized PDB's own chain-L HETATM lines (fixed column widths,
    standard PDB format)."""
    pdb_path = CHIMERAX_MIN_DIR / f"{complex_id}.pdb"
    lookup: dict[tuple[float, float, float], str] = {}
    with open(pdb_path) as f:
        for line in f:
            if line.startswith("HETATM") and line[21] == "L":
                name = line[12:16].strip()
                xyz = (round(float(line[30:38]), 2), round(float(line[38:46]), 2), round(float(line[46:54]), 2))
                lookup[xyz] = name
    return lookup


def _ligand_side_label(atom_lookup: dict, row: dict) -> str:
    """Single ligand atom name for an atom-to-atom interaction (LIGCOO
    matches one real atom exactly); PLIP's own LIG_GROUP label for a
    group-to-atom interaction (LIGCOO is that group's centroid, so no atom
    matches it); a fixed "aromatic_ring" label for pi-stacking specifically
    (always ring-based, but never carries LIG_GROUP)."""
    x, y, z = (round(float(v), 2) for v in row["ligcoo"].split(","))
    atom_name = atom_lookup.get((x, y, z))
    if atom_name is not None:
        return atom_name
    if row.get("lig_group"):
        return row["lig_group"]
    if row.get("type") is not None:  # pi_stacks: T/P-shaped ring stacking, no LIG_GROUP field
        return "aromatic_ring"
    return "?"


def atom_level_interactions(protein: str) -> pd.DataFrame:
    """One row per (complex_id, interaction_type, resnr, ligand_atom)."""
    complex_ids = rosetta.loc[rosetta["protein"] == protein, "complex_id"].unique()
    rows = []
    for cid in complex_ids:
        report_path = PLIP_REPORT_DIR / f"{cid}_report.txt"
        pdb_path = CHIMERAX_MIN_DIR / f"{cid}.pdb"
        if not report_path.exists() or not pdb_path.exists():
            continue
        atom_lookup = _ligand_atom_names(cid)
        for interaction_type, entries in plip2dictlist(report_path).items():
            for row in entries:
                if row.get("reschain", "A") != "A":
                    continue
                rows.append({
                    "complex_id": cid,
                    "interaction_type": interaction_type,
                    "resnr": int(row["resnr"]),
                    "ligand_atom": _ligand_side_label(atom_lookup, row),
                })
    return pd.DataFrame(rows)


def _atom_sort_key(name: str):
    """"C1" < "C4" < "C10" (numeric, not string) order; non-atom labels
    ("Carboxylate", "aromatic_ring", "?") sort after every real atom."""
    import re
    m = re.match(r"([A-Za-z]+)(\d+)", name)
    return (1, name) if m is None else (0, m.group(1), int(m.group(2)))


def plot_interaction_matrices(protein: str):
    df = atom_level_interactions(protein)
    if df.empty:
        print(f"{protein}: no atom-level PLIP data")
        return
    n_complexes = df["complex_id"].nunique()
    for interaction_type in sorted(df["interaction_type"].unique()):
        sub = df[df["interaction_type"] == interaction_type]
        counts = sub.drop_duplicates(["complex_id", "resnr", "ligand_atom"]).groupby(["resnr", "ligand_atom"]).size().reset_index(name="n")
        counts["frequency"] = counts["n"] / n_complexes
        pivot = counts.pivot(index="resnr", columns="ligand_atom", values="frequency").fillna(0)
        pivot = pivot[sorted(pivot.columns, key=_atom_sort_key)]
        fig = px.imshow(
            pivot, color_continuous_scale="Reds", aspect="auto",
            labels=dict(color="frequency"),
            title=f"{protein} -- {interaction_type} (n={n_complexes} complexes, all pose clusters pooled)",
        )
        fig.update_xaxes(title="ligand atom")
        fig.update_yaxes(title="protein residue number", type="category")
        fig.show()


## Interactive: pick one protein


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

_out = widgets.Output()
_dropdown = widgets.Dropdown(options=GA1_PROTEINS, description="protein:")


def _on_change(change):
    with _out:
        clear_output(wait=True)
        protein = change["new"]
        plot_residue_contribution(protein)
        plot_positive_residue_decomposition(protein)
        plot_interaction_types(protein)
        display(combined_summary(protein))
        plot_interaction_matrices(protein)


_dropdown.observe(_on_change, names="value")
display(_dropdown, _out)
_on_change({"new": GA1_PROTEINS[0]})


## Every GA1 protein, one after another

Same plots as above, looped over all 8 GA1 proteins (no dropdown) --
useful for a top-to-bottom scan or for exporting the whole notebook to
HTML. Includes the atom-level matrices, so this cell is slow (a few
minutes total) -- comment out the `plot_interaction_matrices` line if you
just want the fast residue-level plots for every protein.


In [ ]:
for _protein in GA1_PROTEINS:
    plot_residue_contribution(_protein)
    plot_positive_residue_decomposition(_protein)
    plot_interaction_types(_protein)
    plot_interaction_matrices(_protein)
